# MMSD text-only Bi-LSTM comparison baseline

In [1]:
%pip install tqdm==4.67.1


  Attempting uninstall: tqdm
    Found existing installation: tqdm 4.67.3
    Uninstalling tqdm-4.67.3:
      Successfully uninstalled tqdm-4.67.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


## Configuration


In [2]:
import ast
import csv
import json
import os
import pickle
import platform
import random
import time
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

DATA_ROOT = Path('/kaggle/input/datasets/fruich/mmsd-full/data')
ASSET_ROOT = DATA_ROOT / 'raw/mmsd'
SEED = 42
BATCH_SIZE = 32
EPOCHS = 8
PATIENCE = 5
LEARNING_RATE = 0.001
L2 = 1e-7
SEQ_LEN = 75
NUM_WORKERS = 4
RUN_DIR = Path('/kaggle/working/runs') / f'text_baseline_seed{SEED}_single_gpu'
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print('PyTorch:', torch.__version__, 'Device:', DEVICE)


PyTorch: 2.11.0+cu130 Device: cuda:0


## Data and embeddings


In [3]:
class VocabularyLoader(pickle.Unpickler):
    def find_class(self, module, name):
        raise pickle.UnpicklingError('Global references are not allowed in vocabularies')

def load_vocabulary(path):
    with path.open('rb') as stream:
        vocabulary = VocabularyLoader(stream, encoding='latin1').load()
    if not isinstance(vocabulary, dict) or not all(
        isinstance(key, str) and type(value) is int for key, value in vocabulary.items()):
        raise ValueError(f'Invalid vocabulary: {path}')
    return vocabulary

word_vocab = load_vocabulary(ASSET_ROOT / 'words/vocab')
word_weights = torch.from_numpy(np.loadtxt(ASSET_ROOT / 'words/vector', dtype=np.float32))
assert word_weights.shape == (len(word_vocab), 200)
assert word_vocab['<pad>'] == 0
blocked = {'sarcasm', 'sarcastic', 'reposting', '<url>', 'joke', 'humour',
           'humor', 'jokes', 'irony', 'ironic', 'exgag'}

class TextDataset(Dataset):
    def __init__(self, split):
        self.rows = [ast.literal_eval(line) for line in
                     (ASSET_ROOT / f'text/{split}.txt').read_text().splitlines() if line.strip()]
        self.rows = [row for row in self.rows if not blocked.intersection(row[1].split())]
        self.tokens = torch.zeros((len(self.rows), SEQ_LEN), dtype=torch.long)
        self.labels = torch.tensor([row[-1] for row in self.rows], dtype=torch.float32)
        for index, row in enumerate(self.rows):
            ids = [word_vocab.get(token, word_vocab['<unk>']) for token in row[1].split()[:SEQ_LEN]]
            self.tokens[index, :len(ids)] = torch.tensor(ids, dtype=torch.long)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        return self.tokens[index], self.labels[index], str(self.rows[index][0])

train_data = TextDataset('train')
valid_data = TextDataset('valid2')
test_data = TextDataset('test2')
generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
                          num_workers=NUM_WORKERS, pin_memory=DEVICE.type == 'cuda', generator=generator,
                          persistent_workers=NUM_WORKERS > 0)
valid_loader = DataLoader(valid_data, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                          pin_memory=DEVICE.type == 'cuda', persistent_workers=NUM_WORKERS > 0)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                         pin_memory=DEVICE.type == 'cuda', persistent_workers=NUM_WORKERS > 0)
positives = train_data.labels.sum().item()
positive_weight = (len(train_data) - positives) / positives
print('Train / validation / test:', len(train_data), len(valid_data), len(test_data))
print('Positive-class loss weight:', positive_weight)


Train / validation / test: 19816 2410 2409
Positive-class loss weight: 1.292987734320759


## Text-only Bi-LSTM


In [4]:
class TextBiLSTMClassifier(nn.Module):
    def __init__(self, word_weights):
        super().__init__()
        self.word_embedding = nn.Embedding.from_pretrained(word_weights.clone(), freeze=False)
        self.dropout = nn.Dropout(0.2)
        self.lstm = nn.LSTM(200, 256, batch_first=True, bidirectional=True)
        self.classifier = nn.Sequential(nn.Linear(512, 256), nn.ReLU(), nn.Linear(256, 1))
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.trunc_normal_(module.weight, std=0.08, a=-0.16, b=0.16)
                nn.init.constant_(module.bias, 0.01)
        for name, parameter in self.lstm.named_parameters():
            if 'weight' in name:
                nn.init.xavier_uniform_(parameter)
            else:
                nn.init.zeros_(parameter)
                if 'bias_ih' in name:
                    with torch.no_grad():
                        parameter[256:512].fill_(1.0)

    def forward(self, tokens):
        embedded = self.dropout(self.word_embedding(tokens))
        text_raw, _ = self.lstm(embedded)
        return self.classifier(text_raw.mean(dim=1)).squeeze(-1)

    def regularization(self):
        return 0.5 * sum(parameter.square().sum() for name, parameter in self.named_parameters()
                         if parameter.requires_grad and name != 'word_embedding.weight')


## Evaluation


In [5]:
def binary_metrics(labels, probabilities):
    labels = np.asarray(labels, dtype=np.int64)
    predicted = np.asarray(probabilities) > 0.5
    tp = int(((labels == 1) & predicted).sum())
    fp = int(((labels == 0) & predicted).sum())
    tn = int(((labels == 0) & ~predicted).sum())
    fn = int(((labels == 1) & ~predicted).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0
    return {'precision': precision, 'recall': recall, 'f1': f1,
            'accuracy': (tp + tn) / len(labels), 'tp': tp, 'fp': fp, 'tn': tn, 'fn': fn}

@torch.no_grad()
def evaluate(network, loader):
    network.eval()
    labels, probabilities, tweet_ids = [], [], []
    for tokens, targets, ids in tqdm(loader, desc='Evaluate', leave=False):
        logits = network(tokens.to(DEVICE, non_blocking=True))
        probabilities.extend(logits.sigmoid().cpu().tolist())
        labels.extend(targets.tolist())
        tweet_ids.extend(ids)
    return binary_metrics(labels, probabilities), tweet_ids, labels, probabilities


## Training


In [6]:
RUN_DIR.mkdir(parents=True, exist_ok=False)
config = {'model': 'text', 'seed': SEED, 'batch_size': BATCH_SIZE, 'max_epochs': EPOCHS,
          'patience': PATIENCE, 'learning_rate': LEARNING_RATE, 'lr_decay_per_epoch': 0.8,
          'l2': L2, 'sequence_length': SEQ_LEN, 'hidden_size': 256, 'input_dropout': 0.2,
          'positive_weight': positive_weight, 'checkpoint_selection': 'validation positive-class F1',
          'asset_root': str(ASSET_ROOT),
          'split_counts': {'train': len(train_data), 'valid': len(valid_data), 'test': len(test_data)},
          'upstream_revision': 'f42b16510208624d91fa545ca9bb64c6335f971e',
          'python': platform.python_version(), 'torch': torch.__version__, 'numpy': np.__version__,
          'cuda': torch.version.cuda, 'cudnn': torch.backends.cudnn.version(),
          'gpu_names': [torch.cuda.get_device_name(DEVICE)] if DEVICE.type == 'cuda' else [],
          'device': str(DEVICE), 'num_workers': NUM_WORKERS,
          'persistent_workers': NUM_WORKERS > 0, 'non_blocking_transfers': True,
          }
(RUN_DIR / 'config.json').write_text(json.dumps(config, indent=2) + '\n')
base_model = TextBiLSTMClassifier(word_weights).to(DEVICE)
network = base_model
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(positive_weight, device=DEVICE))
optimizer = torch.optim.Adam([p for p in base_model.parameters() if p.requires_grad], lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.8)
best_f1, stale_epochs = -1.0, 0
history = []
start = time.perf_counter()
if DEVICE.type == 'cuda':
    torch.cuda.reset_peak_memory_stats(DEVICE)
for epoch in range(1, EPOCHS + 1):
    network.train()
    epoch_loss, seen = 0.0, 0
    epoch_start = time.perf_counter()
    progress = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS}', unit='batch')
    for tokens, labels, _ in progress:
        tokens, labels = tokens.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(network(tokens), labels) + L2 * base_model.regularization()
        if not torch.isfinite(loss):
            raise FloatingPointError('Non-finite training loss')
        loss.backward()
        nn.utils.clip_grad_norm_(base_model.parameters(), 5.0)
        optimizer.step()
        epoch_loss += loss.item() * len(labels)
        seen += len(labels)
        progress.set_postfix(loss=epoch_loss / seen)
    valid_metrics, _, _, _ = evaluate(network, valid_loader)
    record = {'epoch': epoch, 'train_loss': epoch_loss / seen, 'validation': valid_metrics,
              'learning_rate': optimizer.param_groups[0]['lr'],
              'seconds': time.perf_counter() - epoch_start}
    history.append(record)
    (RUN_DIR / 'history.json').write_text(json.dumps(history, indent=2) + '\n')
    print(json.dumps(record))
    if valid_metrics['f1'] > best_f1:
        best_f1, stale_epochs = valid_metrics['f1'], 0
        temporary = RUN_DIR / 'best.pt.tmp'
        torch.save({'model': base_model.state_dict(), 'epoch': epoch, 'validation_f1': best_f1}, temporary)
        os.replace(temporary, RUN_DIR / 'best.pt')
    else:
        stale_epochs += 1
    scheduler.step()
    if stale_epochs >= PATIENCE:
        break
training_seconds = time.perf_counter() - start
peak_memory = [torch.cuda.max_memory_allocated(DEVICE)] if DEVICE.type == 'cuda' else []
print('Training seconds:', training_seconds, 'Best validation F1:', best_f1)


Epoch 1/8:   0%|          | 0/619 [00:00<?, ?batch/s]

Evaluate:   0%|          | 0/76 [00:00<?, ?it/s]

{"epoch": 1, "train_loss": 0.49490156295515225, "validation": {"precision": 0.794928335170893, "recall": 0.7518248175182481, "f1": 0.7727759914255091, "accuracy": 0.8240663900414937, "tp": 721, "fp": 186, "tn": 1265, "fn": 238}, "learning_rate": 0.001, "seconds": 10.20722766800003}


Epoch 2/8:   0%|          | 0/619 [00:00<?, ?batch/s]

Evaluate:   0%|          | 0/76 [00:00<?, ?it/s]

{"epoch": 2, "train_loss": 0.32794907480863456, "validation": {"precision": 0.7429094236047575, "recall": 0.8467153284671532, "f1": 0.7914230019493177, "accuracy": 0.8224066390041493, "tp": 812, "fp": 281, "tn": 1170, "fn": 147}, "learning_rate": 0.0008, "seconds": 7.08442273199995}


... (11 outputs left out)

{"epoch": 6, "train_loss": 0.13407523026434112, "validation": {"precision": 0.750242013552759, "recall": 0.808133472367049, "f1": 0.7781124497991968, "accuracy": 0.816597510373444, "tp": 775, "fp": 258, "tn": 1193, "fn": 184}, "learning_rate": 0.0003276800000000001, "seconds": 7.075965872999973}


Epoch 7/8:   0%|          | 0/619 [00:00<?, ?batch/s]

Evaluate:   0%|          | 0/76 [00:00<?, ?it/s]

{"epoch": 7, "train_loss": 0.11140455720383487, "validation": {"precision": 0.755859375, "recall": 0.8070907194994786, "f1": 0.7806354009077155, "accuracy": 0.8195020746887967, "tp": 774, "fp": 250, "tn": 1201, "fn": 185}, "learning_rate": 0.0002621440000000001, "seconds": 7.112414349000005}
Training seconds: 52.739632760000006 Best validation F1: 0.7914230019493177


## Final test and saved results


In [7]:
checkpoint = torch.load(RUN_DIR / 'best.pt', map_location=DEVICE, weights_only=True)
base_model.load_state_dict(checkpoint['model'])
test_start = time.perf_counter()
test_metrics, ids, labels, probabilities = evaluate(network, test_loader)
test_seconds = time.perf_counter() - test_start
results = {'model': 'text', 'seed': SEED, 'selected_epoch': checkpoint['epoch'],
           'best_validation_f1': checkpoint['validation_f1'], 'test': test_metrics,
           'training_seconds_including_validation': training_seconds,
           'test_seconds_including_data_loading': test_seconds,
           'peak_allocated_gpu_bytes_per_device': peak_memory,
           'trainable_parameters': sum(p.numel() for p in base_model.parameters() if p.requires_grad)}
(RUN_DIR / 'results.json').write_text(json.dumps(results, indent=2) + '\n')
with (RUN_DIR / 'test_predictions.csv').open('w', newline='') as stream:
    writer = csv.writer(stream)
    writer.writerow(['tweet_id', 'label', 'probability', 'prediction'])
    writer.writerows((tweet_id, int(label), probability, int(probability > 0.5))
                     for tweet_id, label, probability in zip(ids, labels, probabilities))
print(json.dumps(results, indent=2))


Evaluate:   0%|          | 0/76 [00:00<?, ?it/s]

{
  "model": "text",
  "seed": 42,
  "selected_epoch": 2,
  "best_validation_f1": 0.7914230019493177,
  "test": {
    "precision": 0.7195767195767195,
    "recall": 0.8508863399374348,
    "f1": 0.7797419971333015,
    "accuracy": 0.8086342880863429,
    "tp": 816,
    "fp": 318,
    "tn": 1132,
    "fn": 143
  },
  "training_seconds_including_validation": 52.739632760000006,
  "test_seconds_including_data_loading": 0.34022743500003116,
  "peak_allocated_gpu_bytes_per_device": [
    169441280
  ],
  "trainable_parameters": 3525569
}
